# Procesamiento y Curación - History & Biography

Este notebook ejecuta el pipeline de curación para History & Biography y produce datasets Parquet a nivel libro e interacción.

## Decisiones de limpieza

- Los metadatos numéricos llegan como strings y se convierten con `errors='coerce'`.
- Los campos anidados (`authors`, `popular_shelves`, `series`, `similar_books`) se resumen en columnas modelables.
- Las reseñas se limpian de HTML básico y espacios redundantes.
- Los ratings ausentes se separan de ratings explícitos mediante `rating_missing` y `rating_clean`.
- Los outliers se documentan y se generan features capadas al p99 sin modificar los valores originales.

In [ ]:
from pathlib import Path
import os, sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.process_goodreads import process_category
from src.config import CATEGORIES

cfg = CATEGORIES['history_biography']

In [ ]:
report = process_category('history_biography', chunksize=100_000, bucket_count=256)
report

## Validación de salidas

Las validaciones comprueban lectura Parquet, llaves no nulas, ratings limpios y deduplicación básica.

In [ ]:
books = pd.read_parquet(cfg.processed_dir / 'books_curated.parquet')
interactions = pd.read_parquet(cfg.processed_dir / 'interactions_curated.parquet')

assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

books.head(), interactions.head()